In [1]:
import torch
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [2]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="hf_pipeline_sequence_classification_mrpc",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=False,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [3]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding


---[ TableVault Record ]---
---[ TableVault Record ]---



In [4]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

clf = pipeline(
    task="text-classification",
    model=model,
    tokenizer=tokenizer,
)

id2label = {int(k): v for k, v in model.config.id2label.items()}
label2id = {str(k): int(v) for k, v in model.config.label2id.items()}

print(model_name)
print(id2label)
print("pipeline device:", clf.device)


---[ TableVault Record ]---


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}
pipeline device: mps:0
---[ TableVault Record ]---



In [5]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_list(ds)
print(ds)
print(ds[0])

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])

print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())


---[ TableVault Record ]---
Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}
num_examples: 408
positive_rate: 0.6838235294117647
---[ TableVault Record ]---



In [6]:
inputs = [{"text": s1, "text_pair": s2} for s1, s2 in zip(sent1, sent2)]

batch_size = 64
preds = []
scores = []
vault.create_record_list("mrpc_score_predictions", column_names=["prediction", "score"])

with torch.no_grad():
    for i in tqdm(range(0, len(inputs), batch_size)):
        batch_inputs = inputs[i:i + batch_size]
        outputs = clf(
            batch_inputs,
            batch_size=batch_size,
            truncation=True,
            max_length=128,
            function_to_apply="softmax",
        )

        for j, out in enumerate(outputs):
            label = out["label"]
            pred = label2id[label] if label in label2id else int(str(label).split("_")[-1])
            preds.append(pred)
            score = float(out["score"])
            scores.append(score)

            vault.append_record("mrpc_score_predictions", {"prediction": int(pred), "score": score}, 
                           input_items = {"glue_mrpc_validation": [i + j, i + j + 1]}
                           )

y_pred = np.array(preds)
y_score = np.array(scores)
print("done")


---[ TableVault Record ]---


  0%|          | 0/7 [00:00<?, ?it/s]

done
---[ TableVault Record ]---



In [7]:
description = "Per-example inference outputs for the GLUE MRPC validation set produced by the Hugging Face text-classification pipeline using the model textattack/distilbert-base-uncased-MRPC. Each record corresponds to one sentence pair from glue_mrpc_validation and contains two fields: prediction (integer class label, e.g. 0 = not paraphrase, 1 = paraphrase) and score (the model\u2019s softmax confidence for the predicted class). In this workflow, this dataset serves as the stored prediction table used for downstream evaluation, error analysis, and provenance tracking by linking each prediction back to its source validation example."
embedding = get_embeddings(description)
vault.create_description("mrpc_score_predictions", description, embedding)

properties = {"artifact_type": "model predictions", "task": "paraphrase detection", "dataset": "mrpc_score_predictions", "source": "glue/mrpc", "upstream_dataset": "glue_mrpc_validation", "split": "validation", "size": "408", "num_classes": "2", "label_space": "not_paraphrase, paraphrase", "columns": "prediction, score", "score_type": "softmax confidence", "model": "textattack/distilbert-base-uncased-MRPC", "framework": "huggingface transformers pipeline", "input_type": "sentence pair", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_score_predictions", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [8]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))


---[ TableVault Record ]---
{'accuracy': 0.8578431372549019, 'f1': 0.9026845637583892}
                precision    recall  f1-score   support

not_paraphrase       0.89      0.63      0.74       129
    paraphrase       0.85      0.96      0.90       279

      accuracy                           0.86       408
     macro avg       0.87      0.80      0.82       408
  weighted avg       0.86      0.86      0.85       408

---[ TableVault Record ]---



In [9]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])], "score:", float(y_score[i]))


---[ TableVault Record ]---
sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: LABEL_1 score: 0.983514666557312
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: LABEL_0 score: 0.8176295757293701
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 label: LABEL_0 score: 0.7466992139816284
sentence1: The AFL-CIO is waiting until October to decide if it will endorse 

In [10]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "score:", float(y_score[i]))


---[ TableVault Record ]---
num_errors: 58
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1 score: 0.929533839225769
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1 score: 0.8425254225730896
idx: 35
sentence1: Bush wanted " to see an aircraft landing the same way that the pilots saw an aircraft landing , " White House press secretary Ari Fleischer said yesterday .
sentence2: On Tuesday , before Byrd 's speech , Fleischer said Bush wanted ' ' to see an aircraft landing the same way that the pilots saw an aircraft landing .
true: 0 pred: 1 sco

In [11]:
vault.create_record_list("hf_pipeline_sequence_classification_mrpc_summary", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("hf_pipeline_sequence_classification_mrpc_summary", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_score_predictions": [0, len(ds)]
                    })

summary

description = "Aggregate evaluation summary for the MRPC sequence classification run. This dataset contains a single summary record computed by comparing predicted labels from mrpc_score_predictions against ground-truth labels in glue_mrpc_validation. Its fields are: accuracy (float, overall classification accuracy), f1 (float, binary F1 score for paraphrase detection), and classification_report (string, full sklearn classification report with per-class precision, recall, F1, support, and overall averages for not_paraphrase and paraphrase). In this workflow, it serves as the compact experiment-level result table used to document model performance for the Hugging Face pipeline using textattack/distilbert-base-uncased-MRPC on the GLUE MRPC validation set."
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_sequence_classification_mrpc_summary", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "evaluation summary", "source": "glue/mrpc", "split": "validation", "size": "1", "model": "textattack/distilbert-base-uncased-MRPC", "framework": "huggingface transformers pipeline", "input_format": "sentence pair", "output_format": "accuracy, f1, classification report", "label_space": "not_paraphrase, paraphrase", "language": "english", "upstream_predictions_source": "mrpc_score_predictions"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_sequence_classification_mrpc_summary", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---



In [12]:
description = "This notebook evaluates a Hugging Face sequence-classification pipeline for the MRPC paraphrase detection task. It loads the pretrained model textattack/distilbert-base-uncased-MRPC and tokenizer, runs batched inference on the GLUE MRPC validation set, and predicts whether each pair of sentences is a paraphrase. The workflow retrieves the validation data from TableVault, formats sentence pairs for the transformers text-classification pipeline, computes prediction labels and confidence scores, and stores per-example outputs in a TableVault record list linked back to the source dataset rows. It then measures model performance using accuracy, F1, and a full classification report, prints sample predictions and errors for inspection, and saves an experiment-level summary to TableVault. Finally, it attaches natural-language descriptions and embedding-based metadata to the prediction table, summary table, and overall notebook process for experiment tracking and later discovery." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("hf_pipeline_sequence_classification_mrpc", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary sequence classification", "model": "textattack/distilbert-base-uncased-MRPC", "dataset": "glue/mrpc validation", "framework": "huggingface transformers", "inference_api": "transformers pipeline", "tokenizer": "AutoTokenizer", "evaluation": "accuracy, f1-score, classification_report", "output_artifacts": "per-example predictions and summary metrics", "tracking": "tablevault", "embedding_model": "text-embedding-3-large", "hardware": "torch mps or cpu"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("hf_pipeline_sequence_classification_mrpc", cat, embedding, prop)


---[ TableVault Record ]---
---[ TableVault Record ]---

